# OpenVLA MMD layer-selection audit

This notebook **does not train either experimental model**.

It audits the layer-selection statistic before the one-shot institutional run.

The original MMD run was structurally correct after scoring: K=16, selected-layer rank 64,
alpha 32, and exact equality with the uniform baseline's 110,828,288 trainable parameters.
The concern is narrower: the original layer scores were computed from 64 samples/domain,
mean-pooled over the entire multimodal/token sequence, and used a separately chosen median
RBF bandwidth at every layer. The resulting ranking was almost perfectly ordered by depth.

This notebook therefore:

1. Reproduces the original-style score on a larger **task-balanced** sample.
2. Computes a more task-relevant score from the hidden states aligned to **action-token prediction positions**.
3. L2-normalises those action representations and uses the **same multi-kernel RBF bandwidth grid at every layer**.
4. Measures top-16 stability over repeated balanced subsamples.
5. Reports overlap with the original selected set `{0,...,15}` and the correlation with layer depth.
6. Writes a machine-readable selection audit. It does not silently change the experimental selection.

### Decision rule

- If the robust top-16 is substantially the same and boundary layers are stable, keep the selection.
- If it changes materially or boundary layers are unstable, freeze the new robust ranking before the institutional run.
- Either outcome must be reported transparently in the paper.

### Sampling fix in v2

The first audit used `train=False`. OpenVLA maps that to the final 5% of the RLDS training
split when no explicit validation split exists. In the executed LIBERO-Spatial run, that
slice exposed only one task instruction. This version restores `train=True`, matching the
original successful MMD scorer, uses a 1000-frame shuffle buffer to mix tasks, explicitly
collects 16 samples from each of all 10 task instructions, and aborts after 10,000 scanned
samples if balancing cannot be achieved.


## 1. Install the same OpenVLA-facing dependency family used by the successful MMD scoring run

In [1]:
import subprocess
import sys

def install(packages, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages)
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

# Do not reinstall Kaggle's torch / torchvision / numpy.
install([
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "peft==0.11.1",
    "accelerate>=0.25.0",
    "sentencepiece==0.1.99",
    "draccus==0.8.0",
    "json-numpy",
    "protobuf",
])

install([
    "einops",
    "jsonlines",
    "rich",
    "matplotlib",
    "huggingface_hub",
    "wandb",
])

install(["bitsandbytes>=0.45.0"])
install(["tensorflow_datasets==4.9.3"])
install(["git+https://github.com/moojink/dlimp_openvla"], no_deps=True)

print("Dependency installation: PASS")

$ /usr/bin/python3 -m pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 peft==0.11.1 accelerate>=0.25.0 sentencepiece==0.1.99 draccus==0.8.0 json-numpy protobuf
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q einops jsonlines rich matplotlib huggingface_hub wandb


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


$ /usr/bin/python3 -m pip install -q bitsandbytes>=0.45.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.0 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q tensorflow_datasets==4.9.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 40.5 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q --no-deps git+https://github.com/moojink/dlimp_openvla
Dependency installation: PASS


## 2. Pin OpenVLA source and patch the unrelated DROID TensorFlow-Graphics import

In [2]:
import os
import shutil
from pathlib import Path

WORKDIR = Path("/kaggle/working")
OPENVLA_DIR = WORKDIR / "openvla_mmd_audit"
DATA_DIR = WORKDIR / "datasets" / "openvla" / "modified_libero_rlds"
OUTPUT_DIR = WORKDIR / "mmd_selection_audit"
OPENVLA_COMMIT = "c8f03f48af692657d3060c19588038c7220e9af9"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if OPENVLA_DIR.exists():
    shutil.rmtree(OPENVLA_DIR)

subprocess.run(
    ["git", "clone", "https://github.com/openvla/openvla.git", str(OPENVLA_DIR)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", OPENVLA_COMMIT],
    cwd=OPENVLA_DIR,
    check=True,
)

actual = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=OPENVLA_DIR,
    text=True,
).strip()
assert actual == OPENVLA_COMMIT, (actual, OPENVLA_COMMIT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(OPENVLA_DIR), "--no-deps"],
    check=True,
)

patch_file = (
    OPENVLA_DIR / "prismatic" / "vla" / "datasets" / "rlds" /
    "oxe" / "utils" / "droid_utils.py"
)
text = patch_file.read_text()
target = "import tensorflow_graphics.geometry.transformation as tfg"
if target in text:
    patch_file.write_text(text.replace(target, f"# {target}"))

live = [
    line for line in patch_file.read_text().splitlines()
    if line.strip().startswith("import tensorflow_graphics")
]
assert not live, live

print("Pinned OpenVLA:", actual)
print("DROID-only tensorflow_graphics import disabled: PASS")

Cloning into '/kaggle/working/openvla_mmd_audit'...
HEAD is now at c8f03f4 Update README: Remove code block typo


Obtaining file:///kaggle/working/openvla_mmd_audit
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openvla (pyproject.toml): started
  Building editable for openvla (pyproject.toml): finished with status 'done'
  Created wheel for openvla: filename=openvla-0.0.3-0.editable-py3-none-any.whl size=17775 sha256=1f50772fa8f7cae3788b7490365e6478db5ce52ced2e8be60d293bdf73228826
  Stored in directory: /tmp/pip-ephem-wheel-cache-jbv_vldm/wheels/f4/45/c6/1e2b4a4227d784bb0ec03da78a3af6e62775f0de2ccea249e1
Successfully built openvla
P

## 3. Environment and dual-T4 preflight

In [3]:
import torch
import transformers
import tensorflow as tf
import tensorflow_datasets as tfds
import timm
import peft
import accelerate
from dlimp import DLataset

if str(OPENVLA_DIR) not in sys.path:
    sys.path.insert(0, str(OPENVLA_DIR))

from prismatic.vla.datasets.rlds.oxe.utils import droid_utils

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("timm:", timm.__version__)

assert transformers.__version__ == "4.40.1"
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, (
    "This audit intentionally mirrors the successful original scoring path by "
    "placing the FP16 base model across two GPUs."
)

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} "
        f"({props.total_memory / 1024**3:.2f} GiB)"
    )

print("Environment preflight: PASS")

Python: 3.12.13
PyTorch: 2.10.0+cu128 CUDA: 12.8
Transformers: 4.40.1
TensorFlow: 2.20.0
TFDS: 4.9.3
PEFT: 0.11.1
Accelerate: 1.13.0
timm: 0.9.10
GPU 0: Tesla T4 (14.56 GiB)
GPU 1: Tesla T4 (14.56 GiB)
Environment preflight: PASS


## 4. Download only LIBERO-Spatial and LIBERO-Object RLDS subsets

In [4]:
for name in ["libero_spatial_no_noops", "libero_object_no_noops"]:
    target = DATA_DIR / name
    if not target.exists():
        subprocess.run([
            "huggingface-cli", "download",
            "openvla/modified_libero_rlds",
            "--repo-type", "dataset",
            "--include", f"{name}/*",
            "--local-dir", str(DATA_DIR),
        ], check=True)

    if not target.exists():
        raise FileNotFoundError(target)
    print(name, "PASS")

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]Downloading 'libero_spatial_no_noops/1.0.0/libero_spatial-train.tfrecord-00003-of-00016' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_spatial_no_noops/1.0.0/Iknbje4JJDLVtQGvo-9aizL2svQ=.ca5c55cef57390f3e3c8a4ef0cbfe8b155b9f7e5a8b37c993f676faecbe25a3a.incomplete'
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_spatial_no_noops/1.0.0/features.json
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_spatial_no_noops/1.0.0/dataset_info.json
Fetching 18 files:   6%|▌         | 1/18 [00:00<00:03,  4.66it/s]Downloading 'libero_spatial_no_noops/1.0.0/libero_spatial-train.tfrecord-00006-of-00016' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_spatial_no_noops/1.0.0/ApBPOcWrCfiAXFgX0eG5fdwPNOU=.2dfaef9cfc3786ca2b99124f220bb883c654e173fd77a1da2d77d6ae50a16373.

/kaggle/working/datasets/openvla/modified_libero_rlds
libero_spatial_no_noops PASS
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.


Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]Downloading 'libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00002-of-00032' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_object_no_noops/1.0.0/2WsyUB5u-3LgX8uxQpJGyE1rrUY=.faf67f4bbe796620975b9600d0e7f5c3ee49013efd81a5051d38569184e09abf.incomplete'
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/features.json
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/dataset_info.json
Fetching 34 files:   3%|▎         | 1/34 [00:00<00:05,  5.84it/s]Downloading 'libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00006-of-00032' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_object_no_noops/1.0.0/fzWzWPH20Gjd192rzQyt5JZK1qw=.cd3520d52af8f36b3756e9598e16e6c0125dc191713b3a020b0f7b6c87be5e6f.incomple

/kaggle/working/datasets/openvla/modified_libero_rlds
libero_object_no_noops PASS


Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00031-of-00032
Fetching 34 files: 100%|██████████| 34/34 [00:08<00:00,  4.10it/s]


## 5. Write the scoring audit

The action-representation extraction follows OpenVLA's own training alignment:

- model hidden states are sliced from `num_patches:-1`;
- `labels[:, 1:]` identifies the next-token targets;
- only positions whose targets are action tokens are pooled.

That makes the layer statistic much closer to the representations directly used to predict robot actions than
mean-pooling every vision/language/action token together.

In [5]:
from pathlib import Path
import ast

AUDIT_SCRIPT = OUTPUT_DIR / "run_mmd_audit.py"
AUDIT_SCRIPT.write_text('\nimport json\nimport math\nimport os\nimport random\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport tensorflow as tf\nimport torch\nimport torch.nn.functional as F\nfrom scipy.stats import spearmanr\nfrom transformers import AutoModelForVision2Seq, AutoProcessor\n\nsys.path.insert(0, os.environ["OPENVLA_DIR"])\n\nfrom prismatic.vla.datasets import RLDSDataset, RLDSBatchTransform\nfrom prismatic.vla.action_tokenizer import ActionTokenizer\nfrom prismatic.models.backbones.llm.prompting import PurePromptBuilder\n\nDATA_DIR = Path(os.environ["DATA_DIR"])\nOUTPUT_DIR = Path(os.environ["OUTPUT_DIR"])\n\nPER_TASK = 16\nEXPECTED_TASKS = 10\nSUBSAMPLE_PER_TASK = 8\nSTABILITY_REPEATS = 50\nFIXED_SIGMAS = (0.5, 1.0, 2.0)\nSEED = 7\nORIGINAL_SELECTED = set(range(16))\nORIGINAL_SCORES = [\n    0.582154, 0.563868, 0.561108, 0.553814, 0.534216, 0.511073, 0.498595, 0.468621,\n    0.401178, 0.358534, 0.336466, 0.310507, 0.305984, 0.297958, 0.299597, 0.288502,\n    0.280964, 0.280810, 0.273150, 0.268571, 0.254588, 0.250844, 0.226890, 0.215142,\n    0.193935, 0.167999, 0.163774, 0.165174, 0.160612, 0.158327, 0.154718, 0.117524,\n]\n\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\ntf.random.set_seed(SEED)\n\n\nclass AuditBatchTransform(RLDSBatchTransform):\n    def __call__(self, rlds_batch):\n        out = super().__call__(rlds_batch)\n        lang = rlds_batch["task"]["language_instruction"]\n        if hasattr(lang, "decode"):\n            lang = lang.decode()\n        out["language_instruction"] = str(lang).lower()\n        return out\n\n\ndef rbf_kernel(x, y, sigma):\n    dist2 = torch.cdist(x, y, p=2).pow(2)\n    return torch.exp(-dist2 / (2.0 * sigma * sigma))\n\n\ndef unbiased_mmd_from_kernels(kxx, kyy, kxy):\n    m = kxx.shape[0]\n    n = kyy.shape[0]\n    if m < 2 or n < 2:\n        raise ValueError("Need at least two samples per domain")\n    a = (kxx.sum() - torch.trace(kxx)) / (m * (m - 1))\n    b = (kyy.sum() - torch.trace(kyy)) / (n * (n - 1))\n    c = kxy.mean()\n    return (a + b - 2.0 * c).item()\n\n\ndef original_style_mmd(x, y):\n    combined = torch.cat([x, y], dim=0)\n    pdist = torch.pdist(combined, p=2)\n    sigma = torch.median(pdist).item() if pdist.numel() else 1.0\n    sigma = max(float(sigma), 1e-4)\n    return unbiased_mmd_from_kernels(\n        rbf_kernel(x, x, sigma),\n        rbf_kernel(y, y, sigma),\n        rbf_kernel(x, y, sigma),\n    ), sigma\n\n\ndef fixed_mk_mmd(x, y):\n    # Unit normalisation makes a common bandwidth grid meaningful across layers.\n    x = F.normalize(x.float(), p=2, dim=1)\n    y = F.normalize(y.float(), p=2, dim=1)\n\n    kxx = sum(rbf_kernel(x, x, s) for s in FIXED_SIGMAS) / len(FIXED_SIGMAS)\n    kyy = sum(rbf_kernel(y, y, s) for s in FIXED_SIGMAS) / len(FIXED_SIGMAS)\n    kxy = sum(rbf_kernel(x, y, s) for s in FIXED_SIGMAS) / len(FIXED_SIGMAS)\n\n    return unbiased_mmd_from_kernels(kxx, kyy, kxy), (kxx, kyy, kxy)\n\n\nprint("Loading the common pretrained OpenVLA-7B base in FP16 across the two GPUs...")\nmodel = AutoModelForVision2Seq.from_pretrained(\n    "openvla/openvla-7b",\n    torch_dtype=torch.float16,\n    low_cpu_mem_usage=True,\n    trust_remote_code=True,\n    device_map="auto",\n)\nmodel.eval()\n\nprocessor = AutoProcessor.from_pretrained(\n    "openvla/openvla-7b",\n    trust_remote_code=True,\n)\naction_tokenizer = ActionTokenizer(processor.tokenizer)\n\nnum_layers = len(model.language_model.model.layers)\nnum_patches = model.vision_backbone.featurizer.patch_embed.num_patches\nassert num_layers == 32\n\nbatch_transform = AuditBatchTransform(\n    action_tokenizer,\n    processor.tokenizer,\n    image_transform=processor.image_processor.apply_transform,\n    prompt_builder_fn=PurePromptBuilder,\n)\n\n\ndef collect(domain_name):\n    print(f"\\nCollecting balanced activations: {domain_name}")\n\n    dataset = RLDSDataset(\n        DATA_DIR,\n        domain_name,\n        batch_transform,\n        resize_resolution=tuple(model.config.image_sizes),\n        shuffle_buffer_size=1000,\n        train=True,\n        image_aug=False,\n    )\n\n    # Each representation list is keyed by layer, then by instruction.\n    all_token = {i: defaultdict(list) for i in range(num_layers)}\n    action_token = {i: defaultdict(list) for i in range(num_layers)}\n    counts = defaultdict(int)\n\n    max_items = 10_000\n    seen = 0\n\n    with torch.inference_mode():\n        for sample in dataset:\n            seen += 1\n            if seen > max_items:\n                break\n\n            instruction = sample["language_instruction"]\n            if counts[instruction] >= PER_TASK:\n                if len(counts) == EXPECTED_TASKS and all(v >= PER_TASK for v in counts.values()):\n                    break\n                continue\n\n            input_ids = sample["input_ids"].unsqueeze(0)\n            labels = sample["labels"].unsqueeze(0)\n            attention_mask = torch.ones_like(input_ids)\n            pixel_values = sample["pixel_values"].unsqueeze(0)\n\n            # The embedding / language-model input device is the model\'s first parameter device.\n            first_device = next(model.parameters()).device\n\n            output = model(\n                input_ids=input_ids.to(first_device),\n                attention_mask=attention_mask.to(first_device),\n                pixel_values=pixel_values.to(first_device, dtype=torch.float16),\n                labels=labels.to(first_device),\n                output_hidden_states=True,\n                use_cache=False,\n            )\n\n            hidden_states = output.hidden_states\n            if hidden_states is None or len(hidden_states) != num_layers + 1:\n                raise RuntimeError(\n                    f"Expected {num_layers + 1} hidden-state tensors, got "\n                    f"{None if hidden_states is None else len(hidden_states)}"\n                )\n\n            action_mask_cpu = (\n                labels[:, 1:] > action_tokenizer.action_token_begin_idx\n            )\n\n            for layer_idx in range(num_layers):\n                h = hidden_states[layer_idx + 1]  # output after decoder layer layer_idx\n\n                # Original method: mean over the complete multimodal/token sequence.\n                all_vec = h.mean(dim=1).squeeze(0).detach().float().cpu()\n\n                # Training-aligned positions: same sequence alignment used by OpenVLA\'s\n                # action-token accuracy calculation.\n                aligned = h[:, num_patches:-1, :]\n                mask = action_mask_cpu.to(aligned.device)\n                if aligned.shape[1] != mask.shape[1]:\n                    raise RuntimeError(\n                        f"Layer {layer_idx}: aligned hidden length {aligned.shape[1]} "\n                        f"!= action mask length {mask.shape[1]}"\n                    )\n\n                selected = aligned[mask]\n                if selected.shape[0] == 0:\n                    raise RuntimeError("No action-token prediction positions found")\n\n                action_vec = selected.mean(dim=0).detach().float().cpu()\n\n                all_token[layer_idx][instruction].append(all_vec)\n                action_token[layer_idx][instruction].append(action_vec)\n\n            counts[instruction] += 1\n\n            if sum(counts.values()) % 20 == 0:\n                print(\n                    f"  collected={sum(counts.values())} "\n                    f"tasks_seen={len(counts)} "\n                    f"min_per_task={min(counts.values())}",\n                    flush=True,\n                )\n\n            if len(counts) == EXPECTED_TASKS and all(v >= PER_TASK for v in counts.values()):\n                break\n\n    if len(counts) != EXPECTED_TASKS or not all(v >= PER_TASK for v in counts.values()):\n        raise RuntimeError(\n            "Balanced collection failed before the scan limit. "\n            f"seen={seen}, expected_tasks={EXPECTED_TASKS}, "\n            f"per_task_target={PER_TASK}, counts={dict(counts)}"\n        )\n    if not all(v == PER_TASK for v in counts.values()):\n        raise RuntimeError(f"Unexpected task counts after balancing: {dict(counts)}")\n\n    instructions = sorted(counts)\n    print("Balanced task counts:")\n    for instruction in instructions:\n        print(f"  {counts[instruction]:2d}  {instruction}")\n\n    def stack(rep):\n        layer_arrays = {}\n        layer_task_slices = {}\n        for layer_idx in range(num_layers):\n            chunks = []\n            slices = {}\n            cursor = 0\n            for instruction in instructions:\n                values = rep[layer_idx][instruction]\n                if len(values) != PER_TASK:\n                    raise RuntimeError(\n                        f"Layer {layer_idx}, task \'{instruction}\': {len(values)} != {PER_TASK}"\n                    )\n                block = torch.stack(values)\n                chunks.append(block)\n                slices[instruction] = list(range(cursor, cursor + len(values)))\n                cursor += len(values)\n            layer_arrays[layer_idx] = torch.cat(chunks, dim=0)\n            layer_task_slices[layer_idx] = slices\n        return layer_arrays, layer_task_slices\n\n    all_arr, all_slices = stack(all_token)\n    action_arr, action_slices = stack(action_token)\n\n    return {\n        "counts": dict(counts),\n        "instructions": instructions,\n        "all": all_arr,\n        "action": action_arr,\n        "slices": action_slices,\n    }\n\n\nsource = collect("libero_spatial_no_noops")\ntarget = collect("libero_object_no_noops")\n\nprint("\\nComputing full-sample scores...")\nrows = []\nkernel_cache = {}\n\nfor layer in range(num_layers):\n    x_all = source["all"][layer]\n    y_all = target["all"][layer]\n    original_like, sigma = original_style_mmd(x_all, y_all)\n\n    x_action = source["action"][layer]\n    y_action = target["action"][layer]\n    robust, kernels = fixed_mk_mmd(x_action, y_action)\n    kernel_cache[layer] = kernels\n\n    rows.append({\n        "layer": layer,\n        "original_like_mmd2": original_like,\n        "original_like_sigma": sigma,\n        "robust_action_mk_mmd2": robust,\n    })\n\n# Stability test: equal per-task subsample, repeated with different RNG seeds.\nprint("\\nRunning balanced-subsample ranking stability...")\nstability_scores = np.zeros((STABILITY_REPEATS, num_layers), dtype=np.float64)\n\nsource_instructions = source["instructions"]\ntarget_instructions = target["instructions"]\n\nfor rep in range(STABILITY_REPEATS):\n    rng = np.random.default_rng(SEED + 1000 + rep)\n\n    source_idx = []\n    for instruction in source_instructions:\n        pool = np.asarray(source["slices"][0][instruction])\n        source_idx.extend(rng.choice(pool, SUBSAMPLE_PER_TASK, replace=False).tolist())\n\n    target_idx = []\n    for instruction in target_instructions:\n        pool = np.asarray(target["slices"][0][instruction])\n        target_idx.extend(rng.choice(pool, SUBSAMPLE_PER_TASK, replace=False).tolist())\n\n    sx = torch.tensor(source_idx, dtype=torch.long)\n    ty = torch.tensor(target_idx, dtype=torch.long)\n\n    for layer in range(num_layers):\n        kxx, kyy, kxy = kernel_cache[layer]\n        sub_xx = kxx[sx][:, sx]\n        sub_yy = kyy[ty][:, ty]\n        sub_xy = kxy[sx][:, ty]\n        stability_scores[rep, layer] = unbiased_mmd_from_kernels(\n            sub_xx, sub_yy, sub_xy\n        )\n\nrobust_full = np.asarray([r["robust_action_mk_mmd2"] for r in rows])\nrobust_order = np.argsort(-robust_full)\nrobust_top16 = set(robust_order[:16].tolist())\n\nrank_matrix = np.zeros_like(stability_scores, dtype=np.int64)\ntop16_frequency = np.zeros(num_layers, dtype=np.float64)\n\nfor rep in range(STABILITY_REPEATS):\n    order = np.argsort(-stability_scores[rep])\n    rank_matrix[rep, order] = np.arange(1, num_layers + 1)\n    top16_frequency[order[:16]] += 1\n\ntop16_frequency /= STABILITY_REPEATS\n\nfor layer in range(num_layers):\n    rows[layer]["robust_rank"] = int(np.where(robust_order == layer)[0][0] + 1)\n    rows[layer]["stability_mean_rank"] = float(rank_matrix[:, layer].mean())\n    rows[layer]["stability_std_rank"] = float(rank_matrix[:, layer].std())\n    rows[layer]["top16_inclusion_frequency"] = float(top16_frequency[layer])\n\noriginal_depth_corr = spearmanr(np.arange(num_layers), ORIGINAL_SCORES).statistic\nrobust_depth_corr = spearmanr(np.arange(num_layers), robust_full).statistic\n\noverlap = sorted(ORIGINAL_SELECTED & robust_top16)\njaccard = len(ORIGINAL_SELECTED & robust_top16) / len(ORIGINAL_SELECTED | robust_top16)\n\n# Stable boundary definition: chosen layers should appear in top16 >=80% of repeats,\n# excluded layers should appear <=20%. Anything else near the cutoff is flagged.\nunstable = [\n    i for i, f in enumerate(top16_frequency)\n    if 0.20 < f < 0.80\n]\n\nsummary = {\n    "sampling": {\n        "per_task_per_domain": PER_TASK,\n        "tasks_per_domain": EXPECTED_TASKS,\n        "samples_per_domain": PER_TASK * EXPECTED_TASKS,\n        "stability_repeats": STABILITY_REPEATS,\n        "subsample_per_task_per_repeat": SUBSAMPLE_PER_TASK,\n    },\n    "original_method": {\n        "selected_layers": sorted(ORIGINAL_SELECTED),\n        "depth_spearman": float(original_depth_corr),\n    },\n    "robust_method": {\n        "representation": "mean hidden state at OpenVLA action-token prediction positions",\n        "normalisation": "per-sample L2",\n        "kernel": "equal-weight MK-RBF",\n        "fixed_sigmas": list(FIXED_SIGMAS),\n        "selected_layers": sorted(robust_top16),\n        "depth_spearman": float(robust_depth_corr),\n    },\n    "comparison": {\n        "top16_overlap_count": len(overlap),\n        "top16_overlap_layers": overlap,\n        "top16_jaccard": float(jaccard),\n        "unstable_boundary_layers": unstable,\n    },\n}\n\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nwith open(OUTPUT_DIR / "mmd_selection_audit.json", "w") as f:\n    json.dump({"summary": summary, "layers": rows}, f, indent=2)\n\nwith open(OUTPUT_DIR / "robust_selected_layers.txt", "w") as f:\n    f.write(",".join(map(str, sorted(robust_top16))))\n\nwith open(OUTPUT_DIR / "mmd_selection_audit.csv", "w") as f:\n    headers = list(rows[0].keys())\n    f.write(",".join(headers) + "\\n")\n    for row in rows:\n        f.write(",".join(str(row[h]) for h in headers) + "\\n")\n\nprint("\\n" + "=" * 78)\nprint("MMD SELECTION AUDIT SUMMARY")\nprint("=" * 78)\nprint("Original selected :", sorted(ORIGINAL_SELECTED))\nprint("Robust selected   :", sorted(robust_top16))\nprint("Top-16 overlap    :", len(overlap), "/ 16")\nprint("Jaccard           :", round(jaccard, 4))\nprint("Original depth ρ  :", round(float(original_depth_corr), 6))\nprint("Robust depth ρ    :", round(float(robust_depth_corr), 6))\nprint("Unstable boundary :", unstable)\nprint("\\nLayer | robust rank | top16 freq | robust MMD² | original-like MMD²")\nfor row in sorted(rows, key=lambda r: r["robust_rank"]):\n    print(\n        f"{row[\'layer\']:>5} | "\n        f"{row[\'robust_rank\']:>11} | "\n        f"{row[\'top16_inclusion_frequency\']:.2f}       | "\n        f"{row[\'robust_action_mk_mmd2\']:+.6f}   | "\n        f"{row[\'original_like_mmd2\']:+.6f}"\n    )\n\nprint("\\nSaved:")\nprint(OUTPUT_DIR / "mmd_selection_audit.json")\nprint(OUTPUT_DIR / "mmd_selection_audit.csv")\nprint(OUTPUT_DIR / "robust_selected_layers.txt")\n')
ast.parse(AUDIT_SCRIPT.read_text())

print("Audit script:", AUDIT_SCRIPT)
print("Syntax validation: PASS")

Audit script: /kaggle/working/mmd_selection_audit/run_mmd_audit.py
Syntax validation: PASS


## 6. Run the audit

In [6]:
env = os.environ.copy()
env.update({
    "OPENVLA_DIR": str(OPENVLA_DIR),
    "DATA_DIR": str(DATA_DIR),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "TOKENIZERS_PARALLELISM": "false",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})

cmd = [sys.executable, "-u", str(AUDIT_SCRIPT)]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, env=env)

$ /usr/bin/python3 -u /kaggle/working/mmd_selection_audit/run_mmd_audit.py
Loading the common pretrained OpenVLA-7B base in FP16 across the two GPUs...


Loading checkpoint shards: 100%|██████████| 3/3 [01:00<00:00, 20.30s/it]



09/30 [11:55:00] INFO     | >> [*] Computing dataset           data_utils.py:232
                          statistics. This may take a bit, but                  
                          should only need to happen once.                      


100%|██████████| 432/432 [00:07<00:00, 57.25it/s]



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# libero_spatial_no_noops: =================================================1.000000 #
######################################################################################

09/30 [11:55:08] INFO     | >> [*] Threads per Dataset: [1]       dataset.py:531
                 INFO     | >> [*] Reads per Dataset: [1]         dataset.py:532
                 INFO     | >> [*] Constructing datasets...       dataset.py:535
09/30 [11:55:09] INFO     | >> [*] Applying frame transforms on   dataset.py:575
                          dataset...                                            
  collected=20 tasks_seen=6 min_per_task=1
  collected=40 tasks_seen=6 min_per_task=4
  collected=60 tasks_seen=6 min_per_task=7
  collected=80 tasks_seen=6 min_per_task=9
  collected=100 tasks_seen=7 min_per_task=4
  collected=120 tasks_seen=8 m

100%|██████████| 454/454 [00:11<00:00, 40.84it/s]



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# libero_object_no_noops: ==================================================1.000000 #
######################################################################################

09/30 [11:56:24] INFO     | >> [*] Threads per Dataset: [1]       dataset.py:531
                 INFO     | >> [*] Reads per Dataset: [1]         dataset.py:532
                 INFO     | >> [*] Constructing datasets...       dataset.py:535
09/30 [11:56:25] INFO     | >> [*] Applying frame transforms on   dataset.py:575
                          dataset...                                            
  collected=20 tasks_seen=7 min_per_task=1
  collected=40 tasks_seen=7 min_per_task=3
  collected=60 tasks_seen=8 min_per_task=1
  collected=80 tasks_seen=8 min_per_task=1
  collected=100 tasks_seen=8 min_per_task=2
  collected=120 tasks_seen=8 m

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/mmd_selection_audit/run_mmd_audit.py'], returncode=0)

## 7. Display the final selection audit

In [7]:
import json
import pandas as pd
from IPython.display import display

result = json.loads((OUTPUT_DIR / "mmd_selection_audit.json").read_text())
summary = result["summary"]

print(json.dumps(summary, indent=2))

df = pd.DataFrame(result["layers"]).sort_values("robust_rank")
display(df)

print("\nFiles to send back:")
print(OUTPUT_DIR / "mmd_selection_audit.json")
print(OUTPUT_DIR / "mmd_selection_audit.csv")

{
  "sampling": {
    "per_task_per_domain": 16,
    "tasks_per_domain": 10,
    "samples_per_domain": 160,
    "stability_repeats": 50,
    "subsample_per_task_per_repeat": 8
  },
  "original_method": {
    "selected_layers": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15
    ],
    "depth_spearman": -0.999266862170088
  },
  "robust_method": {
    "representation": "mean hidden state at OpenVLA action-token prediction positions",
    "normalisation": "per-sample L2",
    "kernel": "equal-weight MK-RBF",
    "fixed_sigmas": [
      0.5,
      1.0,
      2.0
    ],
    "selected_layers": [
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      25
    ],
    "depth_spearman": 0.25806451612903225
  },
  "comparison": {
    "top16_overlap_count": 7,
    "top16_overlap_layers": [
      9,


,layer,original_like_mmd2,original_like_sigma,robust_action_mk_mmd2,robust_rank,stability_mean_rank,stability_std_rank,top16_inclusion_frequency
10,10,0.351255,10.569089,0.090747,1,1.58,0.802247,1.00
14,14,0.314420,13.479630,0.090392,2,1.62,0.485386,1.00
15,15,0.307828,14.142446,0.089172,3,2.80,0.400000,1.00
16,16,0.302165,14.640693,0.086324,4,4.58,0.635295,1.00
12,12,0.322522,12.259725,0.086303,5,4.50,0.500000,1.00
11,11,0.325136,11.796721,0.084558,6,6.00,0.447214,1.00
13,13,0.314245,12.578451,0.083269,7,6.96,0.195959,1.00
17,17,0.302530,14.749143,0.080992,8,7.96,0.280000,1.00
18,18,0.296752,15.008603,0.076902,9,9.00,0.000000,1.00
19,19,0.292462,15.304667,0.072623,10,10.00,0.000000,1.00



Files to send back:
/kaggle/working/mmd_selection_audit/mmd_selection_audit.json
/kaggle/working/mmd_selection_audit/mmd_selection_audit.csv


## What happens next

Do **not** train the final baseline or MMD model from this notebook.

Send back the executed notebook or `mmd_selection_audit.json`.

Once the layer ranking is frozen, the institutional-compute pipeline will:

1. inspect the available GPU model(s), VRAM, BF16 support, RAM and disk;
2. choose the safest non-quantised/BF16 configuration that actually fits;
3. verify both LoRA conditions against the exact same 110,828,288-parameter budget;
4. run a short forward/backward preflight for both conditions before committing to the long jobs;
5. train uniform and MMD from the same untouched OpenVLA-7B base, with identical data/order/optimiser settings;
6. checkpoint adapters frequently and write resumable training state;
7. run real-LIBERO observation sensitivity checks;
8. evaluate both checkpoints through the already validated LIBERO-Object evaluator;
9. emit the final comparison and all paper-ready logs.